In [1]:
import pandas as pd
import json
import time
import random
from bs4 import BeautifulSoup
import datetime
import shutil
import os
import requests

In [ ]:
# olx got anti-bot protection so it blocks requests from normal http libraries like requests
# so we will use FlareSolverr to bypass that protection


In [ ]:
url = "http://localhost:8191/v1"
headers = {"Content-Type": "application/json"}
data = {
    "cmd": "request.get",
    "url": "https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/used/?page=1",
    "maxTimeout": 90000
}
response = requests.post(url, headers=headers, json=data)
print(response.status_code)
print(response.text)

200
{"status": "ok", "message": "Challenge not detected!", "solution": {"url": "https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/used/?page=43", "status": 200, "cookies": [{"domain": ".dubizzle.com.eg", "httpOnly": true, "name": "_cfuvid", "path": "/", "sameSite": "None", "secure": true, "value": "zR.JcCShkNjolWRMnkWtoswcFVfQOab5M8ROHw0kdsU-1763727799735-0.0.1.1-604800000"}], "userAgent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36", "headers": {}, "response": "<html><head></head><body>\n        <script type=\"text/javascript\">\n            !function(e,t){\"object\"==typeof exports&&\"object\"==typeof module?module.exports=t():\"function\"==typeof define&&define.amd?define([],t):\"object\"==typeof exports?exports.Fingerprint=t():e.Fingerprint=t()}(self,(()=>(()=>{\"use strict\";var e={795:function(e,t,n){var r=this&&this.__assign||function(){return r=Object.assign||function(e){for(var t,n=1,r=arguments.length;n<r;n++)for(

In [ ]:
# our status is 200 so the request was successful
# now we can parse the response content with BeautifulSoup

In [ ]:
html = response.json()['solution']['response']
soup = BeautifulSoup(html, "html.parser")
soup.find_all('a',href=True)


In [ ]:
car_links = []

In [95]:
for a in soup.find_all('a', href=True):
    link = a['href']
    if link.startswith('/en/ad/'):
        full_url = 'https://www.dubizzle.com.eg' + link
        car_links.append(full_url)



In [99]:
print(len(car_links))

15548


In [98]:
len(set(car_links))

6764

In [ ]:
# our code works well for first page now we will loop through all pages to collect all car links (200 pages)

In [ ]:
# we loop through pages 1 to 199
# we try each page up to 10 times if no links are found if not we go to the next page
# we get links that start with /en/ad/ (means an add listing) then append it to our list with full url
for page in range(1, 200):
    success = False
    retries = 0
    while not success and retries < 10:
        url = "http://localhost:8191/v1"
        headers = {"Content-Type": "application/json"}
        target_url = f"https://www.dubizzle.com.eg/en/vehicles/cars-for-sale/used/?page={page}"
        data = {
            "cmd": "request.get",
            "url": target_url,
            "maxTimeout": 90000
        }
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 200:
            html = response.json()['solution']['response']
            soup = BeautifulSoup(html, "html.parser")
            links = [a['href'] for a in soup.find_all('a', href=True) if a['href'].startswith('/en/ad/')]
            if links:
                for link in links:
                    full_url = 'https://www.dubizzle.com.eg' + link
                    car_links.append(full_url)
                print(f"Page {page}: {len(links)} car links found.")
                success = True
            else:
                print(f"Page {page}: No links found, retrying...")
                retries += 1
                time.sleep(2)
        else:
            print(f"Page {page}: Request failed ({response.status_code}), retrying...")
            retries += 1
            time.sleep(2)

print(f"Total car links collected: {len(car_links)}")

In [140]:
len(set(car_links)) 

8519

In [141]:
unqiue_car_links = list(set(car_links))
print(f"Total unique car links collected: {len(unqiue_car_links)}")

Total unique car links collected: 8519


In [142]:
pd.DataFrame(unqiue_car_links, columns=['car_link']).to_csv('car_links_unique.csv', index=False)

In [143]:
df=pd.read_csv('car_links_unique.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8519 entries, 0 to 8518
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   car_link  8519 non-null   object
dtypes: object(1)
memory usage: 66.7+ KB


In [ ]:
# we now have all car links we can now scrape each car link for details like price year model etc